# Residual TCN, streaming версия

Эта версия не читает весь sequence parquet в RAM.

Она читает parquet кусками через `iter_batches`, собирает mini-batch пользователей и сразу отдаёт его в TCN.


In [ ]:
import gc
from copy import deepcopy

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import torch
from torch import nn
from tqdm.auto import tqdm


In [ ]:
GAMMA = 1.1

RESIDUAL_FILE = "/kaggle/input/datasets/mu2kagg/sequenceozontech/cnn_residual_target_hurdle_gamma_1_1.csv"

SEQUENCE_NOV = "/kaggle/input/datasets/mu2kagg/sequenceozontech/sequence_cutoff2025-11-15_features.parquet"
SEQUENCE_DEC = "/kaggle/input/datasets/mu2kagg/sequenceozontech/sequence_cutoff2025-12-15_features.parquet"
SEQUENCE_JAN = "/kaggle/input/datasets/mu2kagg/sequenceozontech/sequence_cutoff2026-01-14_features.parquet"
SEQUENCE_TEST = "/kaggle/input/datasets/mu2kagg/sequenceozontech/sequence_cutoff2026-02-13_features.parquet"

CLASSIFIER_TEST_FILE = "/kaggle/input/datasets/mu2kagg/sequenceozontech/catboost_classifier_test_probability.csv"
POSITIVE_REGRESSOR_TEST_FILE = "/kaggle/input/datasets/mu2kagg/sequenceozontech/catboost_competition_test_predictions.csv"

TARGET_COLUMN = "cnn_target_residual_clipped"

HIDDEN_SIZE = 128
BATCH_SIZE = 128
PARQUET_BATCH_SIZE = 250_000
EPOCHS = 10
PATIENCE = 3
LEARNING_RATE = 0.001

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)


In [ ]:
def get_snapshot_name(sequence_file):
    if "2025-11-15" in sequence_file:
        return "train_nov_predict_from_2025-11-15"
    if "2025-12-15" in sequence_file:
        return "train_dec_predict_from_2025-12-15"
    if "2026-01-14" in sequence_file:
        return "train_jan_predict_from_2026-01-14"
    if "2026-02-13" in sequence_file:
        return "competition_test"

    raise ValueError(f"Не знаю snapshot_name для файла: {sequence_file}")


def get_feature_columns(sequence_file):
    columns = pq.ParquetFile(sequence_file).schema_arrow.names

    technical_columns = {
        "user_id",
        "event_date",
        "day_index",
        "target",
    }

    feature_columns = []
    for column in columns:
        if column not in technical_columns:
            feature_columns.append(column)

    return feature_columns


In [ ]:
def read_residual_for_sequence(sequence_file, residual_file, target_column):
    snapshot_name = get_snapshot_name(sequence_file)

    residual = pd.read_csv(
        residual_file,
        usecols=[
            "user_id",
            "snapshot_name",
            "target",
            "target_log",
            "base_hurdle_log",
            target_column,
        ],
    )

    residual = residual.loc[
        residual["snapshot_name"].eq(snapshot_name)
    ].copy()

    if len(residual) == 0:
        raise ValueError(f"Нет residual target для {snapshot_name}")

    return residual


def make_target_dict(sequence_file, residual_file, target_column):
    residual = read_residual_for_sequence(
        sequence_file=sequence_file,
        residual_file=residual_file,
        target_column=target_column,
    )

    target_by_user = dict(
        zip(
            residual["user_id"].to_numpy(),
            residual[target_column].to_numpy(dtype=np.float32),
        )
    )

    del residual
    gc.collect()

    return target_by_user


In [ ]:
def read_ready_parquet_batches(sequence_file, columns_to_read, parquet_batch_size):
    parquet_file = pq.ParquetFile(sequence_file)
    saved_tail = None

    for arrow_batch in parquet_file.iter_batches(
            batch_size=parquet_batch_size,
            columns=columns_to_read,
    ):
        batch = arrow_batch.to_pandas()

        if saved_tail is not None:
            batch = pd.concat([saved_tail, batch], ignore_index=True)

        last_user = batch["user_id"].iloc[-1]
        last_user_mask = batch["user_id"].eq(last_user)

        ready = batch.loc[~last_user_mask].copy()
        saved_tail = batch.loc[last_user_mask].copy()

        if len(ready) > 0:
            yield ready

        del batch
        del ready
        gc.collect()

    if saved_tail is not None and len(saved_tail) > 0:
        yield saved_tail


In [ ]:
def iter_user_batches_from_dataframe(
        data,
        feature_columns,
        batch_size,
        target_by_user=None,
        shuffle_users=False,
):
    user_values = data["user_id"].to_numpy()
    day_indices = data["day_index"].to_numpy(dtype=np.int64)
    feature_values = data[feature_columns].to_numpy(
        dtype=np.float32,
        copy=False,
    )

    user_changes = np.flatnonzero(user_values[1:] != user_values[:-1]) + 1
    user_starts = np.concatenate(([0], user_changes))
    user_ends = np.concatenate((user_changes, [len(data)]))

    user_order = np.arange(len(user_starts))
    if shuffle_users:
        np.random.shuffle(user_order)

    for batch_start in range(0, len(user_order), batch_size):
        batch_user_order = user_order[batch_start:batch_start + batch_size]

        features_batch = np.zeros(
            (len(batch_user_order), 180, len(feature_columns)),
            dtype=np.float32,
        )
        target_batch = np.zeros(len(batch_user_order), dtype=np.float32)
        user_id_batch = np.zeros(len(batch_user_order), dtype=np.int64)

        for row_number, user_position in enumerate(batch_user_order):
            start = user_starts[user_position]
            end = user_ends[user_position]

            user_id = user_values[start]
            user_id_batch[row_number] = user_id

            real_days = (
                (day_indices[start:end] >= 0)
                & (day_indices[start:end] < 180)
            )
            user_days = day_indices[start:end][real_days]
            user_features = feature_values[start:end][real_days]
            features_batch[row_number, user_days] = user_features

            if target_by_user is not None:
                if user_id not in target_by_user:
                    raise ValueError(f"Нет target для user_id={user_id}")
                target_batch[row_number] = target_by_user[user_id]

        yield (
            torch.from_numpy(features_batch),
            torch.from_numpy(target_batch),
            torch.from_numpy(user_id_batch),
        )


In [ ]:
def iter_sequence_batches(
        sequence_file,
        feature_columns,
        batch_size,
        parquet_batch_size,
        residual_file=None,
        target_column=None,
        shuffle_users=False,
):
    columns_to_read = ["user_id", "day_index"] + feature_columns

    target_by_user = None
    if target_column is not None:
        target_by_user = make_target_dict(
            sequence_file=sequence_file,
            residual_file=residual_file,
            target_column=target_column,
        )

    for data in read_ready_parquet_batches(
            sequence_file=sequence_file,
            columns_to_read=columns_to_read,
            parquet_batch_size=parquet_batch_size,
    ):
        yield from iter_user_batches_from_dataframe(
            data=data,
            feature_columns=feature_columns,
            batch_size=batch_size,
            target_by_user=target_by_user,
            shuffle_users=shuffle_users,
        )

        del data
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


In [ ]:
class TCNBlock(nn.Module):
    def __init__(self, channels, kernel_size=3, dilation=1, dropout=0.1):
        super().__init__()

        padding = (kernel_size - 1) * dilation

        self.conv = nn.Conv1d(
            in_channels=channels,
            out_channels=channels,
            kernel_size=kernel_size,
            padding=padding,
            dilation=dilation,
        )
        self.norm = nn.BatchNorm1d(channels)
        self.activation = nn.GELU()
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        residual = x

        x = self.conv(x)
        x = x[:, :, :residual.size(2)]
        x = self.norm(x)
        x = self.activation(x)
        x = self.dropout(x)

        return x + residual


class ResidualTCNModel(nn.Module):
    def __init__(self, input_size, hidden_size=128, dropout=0.1):
        super().__init__()

        self.input_projection = nn.Conv1d(
            in_channels=input_size,
            out_channels=hidden_size,
            kernel_size=1,
        )

        self.blocks = nn.Sequential(
            TCNBlock(hidden_size, dilation=1, dropout=dropout),
            TCNBlock(hidden_size, dilation=2, dropout=dropout),
            TCNBlock(hidden_size, dilation=4, dropout=dropout),
            TCNBlock(hidden_size, dilation=8, dropout=dropout),
        )

        self.head = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.LayerNorm(hidden_size),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 1),
        )

    def forward(self, x):
        x = x.transpose(1, 2)
        x = self.input_projection(x)
        x = self.blocks(x)
        x = self.head(x)

        return x.squeeze(-1)


In [ ]:
def train_one_epoch(
        model,
        train_files,
        residual_file,
        feature_columns,
        target_column,
        device,
        optimizer,
        criterion,
):
    model.train()
    loss_sum = 0
    users_count = 0

    for train_file in train_files:
        print("Загружаем train", train_file)

        for features, target, user_id in tqdm(
                iter_sequence_batches(
                    sequence_file=train_file,
                    feature_columns=feature_columns,
                    batch_size=BATCH_SIZE,
                    parquet_batch_size=PARQUET_BATCH_SIZE,
                    residual_file=residual_file,
                    target_column=target_column,
                    shuffle_users=True,
                ),
                desc="Train batches",
                leave=False,
        ):
            features = features.to(device)
            target = target.to(device)

            optimizer.zero_grad()
            prediction = model(features)
            loss = criterion(prediction, target)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            loss_sum += loss.item() * len(target)
            users_count += len(target)

    return loss_sum / users_count


def calculate_valid_loss(
        model,
        valid_file,
        residual_file,
        feature_columns,
        target_column,
        device,
        criterion,
):
    model.eval()
    loss_sum = 0
    users_count = 0

    with torch.inference_mode():
        for features, target, user_id in tqdm(
                iter_sequence_batches(
                    sequence_file=valid_file,
                    feature_columns=feature_columns,
                    batch_size=BATCH_SIZE,
                    parquet_batch_size=PARQUET_BATCH_SIZE,
                    residual_file=residual_file,
                    target_column=target_column,
                    shuffle_users=False,
                ),
                desc="Validation batches",
                leave=False,
        ):
            features = features.to(device)
            target = target.to(device)

            prediction = model(features)
            loss = criterion(prediction, target)

            loss_sum += loss.item() * len(target)
            users_count += len(target)

    return loss_sum / users_count


In [ ]:
def train_tcn_with_validation(
        model,
        train_files,
        valid_file,
        residual_file,
        feature_columns,
        target_column,
        epochs,
        device,
        patience,
):
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

    model = model.to(device)
    best_model = deepcopy(model.state_dict())
    best_valid_loss = np.inf
    epochs_without_improvement = 0

    train_losses = []
    valid_losses = []

    for epoch in range(epochs):
        train_loss = train_one_epoch(
            model=model,
            train_files=train_files,
            residual_file=residual_file,
            feature_columns=feature_columns,
            target_column=target_column,
            device=device,
            optimizer=optimizer,
            criterion=criterion,
        )
        valid_loss = calculate_valid_loss(
            model=model,
            valid_file=valid_file,
            residual_file=residual_file,
            feature_columns=feature_columns,
            target_column=target_column,
            device=device,
            criterion=criterion,
        )

        train_losses.append(train_loss)
        valid_losses.append(valid_loss)

        print(
            f"Epoch {epoch + 1}/{epochs} | "
            f"train residual RMSE={train_loss ** 0.5:.5f} | "
            f"valid residual RMSE={valid_loss ** 0.5:.5f}"
        )

        if valid_loss < best_valid_loss:
            best_valid_loss = valid_loss
            best_model = deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= patience:
            print("Остановка на эпохе", epoch + 1)
            break

    model.load_state_dict(best_model)
    model.eval()

    best_epoch = int(np.argmin(valid_losses) + 1)
    return best_epoch, model, train_losses, valid_losses


In [ ]:
def predict_residual(model, sequence_file, feature_columns, device):
    model.eval()

    predictions = []
    user_ids = []

    with torch.inference_mode():
        for features, target, batch_user_ids in tqdm(
                iter_sequence_batches(
                    sequence_file=sequence_file,
                    feature_columns=feature_columns,
                    batch_size=BATCH_SIZE,
                    parquet_batch_size=PARQUET_BATCH_SIZE,
                    shuffle_users=False,
                ),
                desc="Predict batches",
                leave=False,
        ):
            features = features.to(device)
            prediction = model(features)

            predictions.append(prediction.cpu().numpy())
            user_ids.append(batch_user_ids.cpu().numpy())

    return pd.DataFrame(
        {
            "user_id": np.concatenate(user_ids),
            "tcn_residual_prediction": np.concatenate(predictions),
        }
    )


def predict_residual_validation(
        model,
        sequence_file,
        residual_file,
        feature_columns,
        target_column,
        device,
):
    prediction = predict_residual(
        model=model,
        sequence_file=sequence_file,
        feature_columns=feature_columns,
        device=device,
    )

    residual = read_residual_for_sequence(
        sequence_file=sequence_file,
        residual_file=residual_file,
        target_column=target_column,
    )

    prediction = prediction.merge(
        residual[
            [
                "user_id",
                "target",
                "target_log",
                "base_hurdle_log",
                target_column,
            ]
        ],
        on="user_id",
        how="inner",
    )

    return prediction


In [ ]:
def score_residual_alpha(validation_prediction, alpha):
    final_log = (
        validation_prediction["base_hurdle_log"]
        + alpha * validation_prediction["tcn_residual_prediction"]
    )
    final_log = np.maximum(final_log, 0)

    error = validation_prediction["target_log"] - final_log
    return float(np.sqrt(np.mean(error**2)))


def find_best_alpha(validation_prediction, alphas):
    rows = []

    for alpha in alphas:
        rows.append(
            {
                "alpha": alpha,
                "rmsle": score_residual_alpha(validation_prediction, alpha),
            }
        )

    return pd.DataFrame(rows).sort_values("rmsle").reset_index(drop=True)


def get_prediction_column(data):
    if "predict" in data.columns:
        return "predict"
    if "pred_expm1" in data.columns:
        return "pred_expm1"
    if "pred_positive_probability" in data.columns:
        return "pred_positive_probability"

    raise ValueError(f"Не нашел колонку с прогнозом: {data.columns.tolist()}")


def make_submission(residual_prediction, classifier_test_file, positive_regressor_test_file, alpha, output_file):
    classifier = pd.read_csv(classifier_test_file)
    positive_regressor = pd.read_csv(positive_regressor_test_file)

    classifier_predict_column = get_prediction_column(classifier)
    positive_predict_column = get_prediction_column(positive_regressor)

    classifier = classifier[["user_id", classifier_predict_column]].rename(
        columns={classifier_predict_column: "positive_probability"}
    )
    positive_regressor = positive_regressor[
        ["user_id", positive_predict_column]
    ].rename(columns={positive_predict_column: "predict_positive_regressor"})

    submission = classifier.merge(positive_regressor, on="user_id", how="inner")
    submission = submission.merge(residual_prediction, on="user_id", how="inner")

    submission["positive_probability"] = submission["positive_probability"].clip(0, 1)
    submission["base_hurdle_log"] = (
        (submission["positive_probability"] ** GAMMA)
        * np.log1p(submission["predict_positive_regressor"].clip(lower=0))
    )
    submission["final_log"] = (
        submission["base_hurdle_log"]
        + alpha * submission["tcn_residual_prediction"]
    )
    submission["final_log"] = submission["final_log"].clip(lower=0)
    submission["predict"] = np.expm1(submission["final_log"])

    submission[["user_id", "predict"]].to_csv(output_file, index=False)
    print("Saved submission:", output_file)
    print("rows", len(submission))


In [ ]:
feature_columns = get_feature_columns(SEQUENCE_NOV)
input_size = len(feature_columns)

print("input_size", input_size)
print("first features", feature_columns[:10])


In [ ]:
folds = [
    {
        "train_files": [SEQUENCE_NOV],
        "valid_file": SEQUENCE_DEC,
    },
    {
        "train_files": [SEQUENCE_NOV, SEQUENCE_DEC],
        "valid_file": SEQUENCE_JAN,
    },
]

best_epochs = []
validation_predictions = []

for fold_number, fold in enumerate(folds, start=1):
    print("Fold", fold_number)

    model = ResidualTCNModel(
        input_size=input_size,
        hidden_size=HIDDEN_SIZE,
        dropout=0.1,
    )

    best_epoch, model, train_losses, valid_losses = train_tcn_with_validation(
        model=model,
        train_files=fold["train_files"],
        valid_file=fold["valid_file"],
        residual_file=RESIDUAL_FILE,
        feature_columns=feature_columns,
        target_column=TARGET_COLUMN,
        epochs=EPOCHS,
        device=device,
        patience=PATIENCE,
    )

    best_epochs.append(best_epoch)

    fold_prediction = predict_residual_validation(
        model=model,
        sequence_file=fold["valid_file"],
        residual_file=RESIDUAL_FILE,
        feature_columns=feature_columns,
        target_column=TARGET_COLUMN,
        device=device,
    )
    fold_prediction["fold"] = fold_number
    validation_predictions.append(fold_prediction)

    fold_prediction.to_csv(
        f"/kaggle/working/tcn_residual_valid_fold_{fold_number}.csv",
        index=False,
    )

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

validation_predictions = pd.concat(validation_predictions, ignore_index=True)
validation_predictions.to_csv(
    "/kaggle/working/tcn_residual_oof_predictions.csv",
    index=False,
)


In [ ]:
alphas = [
    -0.20,
    -0.15,
    -0.10,
    -0.075,
    -0.05,
    -0.025,
    0.00,
    0.025,
    0.05,
    0.075,
    0.10,
    0.125,
    0.15,
    0.175,
    0.20,
    0.25,
    0.30,
]

alpha_result = find_best_alpha(validation_predictions, alphas)
alpha_result.to_csv("/kaggle/working/tcn_alpha_validation.csv", index=False)
alpha_result


In [ ]:
best_alpha = float(alpha_result.iloc[0]["alpha"])
final_epochs = int(round(np.mean(best_epochs)))
final_epochs = max(final_epochs, 1)

print("best_alpha", best_alpha)
print("final_epochs", final_epochs)


In [ ]:
final_model = ResidualTCNModel(
    input_size=input_size,
    hidden_size=HIDDEN_SIZE,
    dropout=0.1,
)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(final_model.parameters(), lr=LEARNING_RATE)
final_model = final_model.to(device)

for epoch in range(final_epochs):
    train_loss = train_one_epoch(
        model=final_model,
        train_files=[SEQUENCE_NOV, SEQUENCE_DEC, SEQUENCE_JAN],
        residual_file=RESIDUAL_FILE,
        feature_columns=feature_columns,
        target_column=TARGET_COLUMN,
        device=device,
        optimizer=optimizer,
        criterion=criterion,
    )
    print(
        f"Final epoch {epoch + 1}/{final_epochs} | "
        f"train residual RMSE={train_loss ** 0.5:.5f}"
    )


In [ ]:
test_residual_prediction = predict_residual(
    model=final_model,
    sequence_file=SEQUENCE_TEST,
    feature_columns=feature_columns,
    device=device,
)

test_residual_prediction.to_csv(
    "/kaggle/working/tcn_residual_test_prediction.csv",
    index=False,
)

make_submission(
    residual_prediction=test_residual_prediction,
    classifier_test_file=CLASSIFIER_TEST_FILE,
    positive_regressor_test_file=POSITIVE_REGRESSOR_TEST_FILE,
    alpha=best_alpha,
    output_file="/kaggle/working/submission_hurdle_tcn_residual.csv",
)
